# Langchain

## Import Libraries

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
import os
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec



## Local Ollama call

In [2]:
chat = ChatOllama(model="gpt-oss:20b-cloud",
                  base_url="http://host.docker.internal:11434",
                  temperature=0.3,)


### Simple call

In [ ]:
response = chat.invoke("Hello!")
print(response.content)

Hello! 👋 How can I assist you today?


## Local Ollama call with structured input message

In [ ]:
messages = [SystemMessage(content="You are an expert data scientist"),
            HumanMessage(content="Write a Python script that trains a neural network on simulated data")
            ]

response = chat.invoke(messages)
print(response.content)

Below is a **stand‑alone Python script** that

1. **Simulates a toy classification dataset** (two Gaussian blobs in 2‑D).  
2. Builds a small **feed‑forward neural network** with PyTorch.  
3. Trains the network on the synthetic data.  
4. Reports training loss and test accuracy.

Feel free to tweak the hyper‑parameters, network depth, or the data generation logic to suit your needs.

```python
#!/usr/bin/env python3
"""
train_simulated_nn.py

A minimal example that trains a neural network on synthetic data.
"""

# --------------------------------------------------------------------------- #
# Imports
# --------------------------------------------------------------------------- #
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# ---------------------------------------------------

## Local Ollama call with PromptTemplate: old models

In [5]:
# useful for older models such as text-davinci-003, flan-t5, etc. It expects a plain text or s string.
template = PromptTemplate(input_variables=["topic"],
                          template="Explain {topic} in one sentence.")

prompt = template.format(topic="large language models")
response = chat.invoke(prompt)
print(prompt)
print(response.content)

Explain large language models in one sentence.
Large language models are AI systems trained on massive amounts of text to learn statistical patterns of language, enabling them to predict, generate, and understand human-like text.


## Local Ollama call with ChatPromptTemplate: Modern models

In [6]:
# Useful for modern models (such as GPT-4o, GPT-5, Claude,Gemini, Llama 3, Qwen, Gemma, Mistral) 
# that expect MESSAGES instead of a string: MESSAGES = [{"role":"system","content":"You are an expert data scientist"},
# {"role":"user","content":"Explain large language models in one sentence"}]

prompt = ChatPromptTemplate.from_messages([("system", "You are an expert data scientist."),
                                           ("human", "Explain {topic} in one sentence.")
                                           ])

messages = prompt.invoke({"topic": "large language models"})

chain = prompt | chat

response = chain.invoke({"topic": "large language models"})
print(f"MESSAGES = {messages}")
print(f"Response: {response.content}")

MESSAGES = messages=[SystemMessage(content='You are an expert data scientist.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain large language models in one sentence.', additional_kwargs={}, response_metadata={})]
Response: Large language models are sophisticated neural networks trained on massive text corpora to predict and generate human‑like language by learning statistical patterns and contextual relationships.


## Two chains

In [29]:
# Chain 1
prompt1 = ChatPromptTemplate.from_messages([("system", "You are an expert machine learning scientist."),
                                            ("human", "Explain the machine learning concept {topic}.")
                                            ])
# Chain 2
prompt2 = ChatPromptTemplate.from_messages([("human",
                                             "Turn the following concept description into an explanation suitable for a five year old in about 500 words:\n\n{concept}")
                                             ])
parser = StrOutputParser()

chain1 = prompt1 | chat | parser

chain2 = prompt2 | chat | parser

overall_chain = (chain1
                 | RunnableLambda(lambda text: {"concept": text})
                 | chain2)
result = overall_chain.invoke({"topic": "RAG"})

print(result)

**Imagine a very clever robot helper that loves to answer questions.**  
But this robot is a little different from the ones you see in cartoons. It has two best friends that help it do its job: **the Finder** and **the Writer**.

---

### 1. The Finder (Retrieval)

The Finder is like a super‑fast librarian.  
When you ask the robot a question, the Finder looks through a huge pile of books, articles, and notes that the robot has stored somewhere far away.  
It pulls out the parts of the books that are most useful for answering your question.  
Think of it like pulling out the right picture from a picture book when you want to know what a giraffe looks like.

---

### 2. The Writer (Generation)

The Writer is the robot’s brain that talks.  
After the Finder gives it the right pages, the Writer reads those pages and then writes a short, friendly answer that you can understand.  
It uses the information the Finder gave it, so the answer is usually true and not made up.

---

### 3. How The

## Chunking text

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                               chunk_overlap=0,)

### Chunk into documents

In [9]:
docs = text_splitter.create_documents([result])

### Chunk into strings

In [10]:
text_chunks = text_splitter.split_text(result)

## Embeedings

In [11]:
# ollama embeeding models are not responding dur to insuficeint host machine spec.
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text:latest",
                                     base_url="http://host.docker.internal:11434")

# working alternative: hugging face open source embeeding models
hg_face_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Embeeding texts

In [13]:
query_result = hg_face_embeddings.embed_query(text_chunks[0])
print(query_result)

[-0.08756695687770844, -0.0056003243662416935, 0.03092968836426735, -0.011575821787118912, -0.05555087700486183, 0.01145117450505495, 0.005251600407063961, -0.03112175688147545, -0.05049384757876396, -0.01678083837032318, 0.016662966459989548, 0.029413266107439995, -0.006670808885246515, 0.06148575618863106, -0.11306894570589066, 0.0420115664601326, -0.09645149856805801, 0.04151210933923721, -0.08839728683233261, -0.1029147133231163, -0.015690302476286888, 0.03147082030773163, 0.0421367771923542, 0.02172926813364029, 0.06655263900756836, 0.04126298055052757, -0.0446472242474556, -0.04436326399445534, 0.003737304825335741, 0.03419975936412811, 0.01267902459949255, 0.02861364558339119, 0.05257883295416832, 0.027844756841659546, 0.0006630631978623569, 0.0019370951922610402, -0.02589845471084118, 0.06113632768392563, -0.04452255368232727, -0.027410242706537247, -0.027426498010754585, -0.07682710140943527, -0.01974262297153473, -0.02714276686310768, 0.02515493333339691, 0.017065975815057755

### Embeeding docs

In [15]:
embeding_vectors = hg_face_embeddings.embed_documents([doc.page_content for doc in docs])

print(len(embeding_vectors))
print(len(embeding_vectors[0]))

56
384


## Pinecone

### Initialize

In [16]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "langchain-quickstart"

### Create pinecone index

In [ ]:
pc.create_index(name=index_name,
                dimension=384,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws",region="us-east-1"))

{
    "name": "langchain-quickstart",
    "metric": "cosine",
    "host": "langchain-quickstart-04txlx7.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

### TEXTS: Pinecone upload and store

In [26]:
vector_store = PineconeVectorStore.from_texts(texts=text_chunks,
                                              embedding=hg_face_embeddings,
                                              index_name=index_name)

### DOCS: Pinecone upload and store

In [25]:
vector_store = PineconeVectorStore.from_documents(documents=docs,
                                                  embedding=hg_face_embeddings,
                                                  index_name=index_name)

### Vector similarity search 

In [ ]:
query = "What is the advantage of the RAG?"
result = vector_store.similarity_search(query)
print(result)

[Document(id='e46a9613-ec8b-4d79-9777-64333ef0deb9', metadata={}, page_content='An autoencoder is like a clever robot that learns to shrink a picture into a tiny secret code and'), Document(id='609ba151-a226-4343-bb75-e5367f8cd570', metadata={}, page_content='An autoencoder is like a clever robot that learns to shrink a picture into a tiny secret code and'), Document(id='408f5df5-2b95-44e9-941d-153ac157110d', metadata={}, page_content='**Autoencoders – a story for a five‑year‑old**'), Document(id='67f92ac9-40f3-4409-af5e-ac430eae0e61', metadata={}, page_content='**Autoencoders – a story for a five‑year‑old**')]
